# Modelamiento Geoquímico Computacional del Valle Medio del Magdalena (VMM)
## Evaluación del Potencial de Almacenamiento Geológico de CO₂
 
---
 
| | |
|---|---|
| **Estudiante** | Danna Hernández |
| **Profesor** | Carlos Pinilla |
| **Programa** | Geologìa |
| **Asignatura** | Modelamiento Computacional |
| **Entrega** | Informe Final  |
| **Software** | PHREEQC v3 · Ostrich |
 
---
 
> Las simulaciones presentadas en este repositorio corresponden a ejercicios de referencia base en el marco de la fase exploratoria del proyecto. Los parámetros empleados fueron construidos a partir de datos de literatura especializada, expedientes del ANLA y registros previos del Valle Medio del Magdalena. 
> Los resultados deben interpretarse como umbrales de comportamiento geoquímico y herramientas de sensibilidad paramétrica, no como pronósticos definitivos del sistema real.

## 1. Resumen Ejecutivo
 
El presente proyecto se enmarca en la evaluación del potencial de almacenamiento geológico de CO₂ en
formaciones del Valle Medio del Magdalena (VMM), Colombia, utilizando modelamiento geoquímico computacional
con PHREEQC versión 3. La investigación constituye una contribución a la caracterización de cuencas
sedimentarias colombianas para captura y almacenamiento de carbono (CCS, *Carbon Capture and Storage*),
tecnología considerada fundamental en los escenarios de mitigación de cambio climático del IPCC.
 
El trabajo aborda cuatro componentes metodológicos articulados: (i) construcción de la base de datos
fisicoquímica desde cero mediante síntesis bibliográfica sistemática y consulta de expedientes regulatorios;
(ii) especiación geoquímica de aguas de formación en cuatro unidades estratigráficas del VMM; (iii)
modelamiento del equilibrio agua-roca y cuantificación de la transferencia de masa mineral; y (iv)
calibración computacional de parámetros petrofísicos mediante el algoritmo de optimización DDS (*Dynamically
Dimensioned Search*) implementado en Ostrich.

## 2. Contexto Geológico – Valle Medio del Magdalena
 
### 2.1 Marco Geotectónico
 
El Valle Medio del Magdalena (VMM) es una cuenca sedimentaria de tipo ante-arco a retroarco ubicada entre
las cordilleras Central y Oriental de Colombia, con una extensión aproximada de 34,000 km². La secuencia
estratigráfica abarca desde el Cretáceo hasta el Neógeno, con espesores sedimentarios que superan los 8,000
metros en los depocentros principales. Su historia de subsidencia térmica y tectónica lo posiciona como una
de las cuencas con mayor potencial para almacenamiento de CO₂ en el norte de Sudamérica.

### 2.2 Formaciones de Interés para CCS
 
| Formación | Edad | Prof. (m) | Litología Dominante | Porosidad (%) | TDS Agua (mg/L) |
|---|---|---|---|---|---|
| La Luna | Cretáceo Superior | 2,503–2,913 | Lutitas calcáreas, chert, carbonatos | 3–8 | ~68,600 |
| Mugrosa, Colorado y Esmeraldas | Eoceno | 793–1,951 | Areniscas y lodolitas fluviales | 8–26 | ~26,500 |
| Tablazo y Rosablanca | Cretáceo Inferior | 3,869 | Calizas y dolomitas | 4–12 | ~94,600 |
| Umir | Paleoceno | 1,311 | Areniscas, lutitas carbonosas | 10–18 | ~32,100 |

### 2.3 Condiciones de Reservorio
 
Las formaciones evaluadas presentan gradientes geotérmicos de 25–35 °C/km y gradientes de presión
hidrostática de 10–11 MPa/km. Para una inyección de CO₂ a profundidades superiores a 800 m, el CO₂ se
encontraría en estado supercrítico (T > 31.1 °C, P > 7.38 MPa), condición necesaria para maximizar la
densidad del fluido y la capacidad volumétrica de almacenamiento.
 
### 2.4 Mineralogía Relevante para entrampamiento Mineral de CO₂
 
La reactividad mineral frente a la inyección de CO₂ determina la capacidad de **entrampamiento mineral** a largo
plazo. Las fases mineralógicas identificadas en los núcleos y análisis XRD disponibles incluyen:
 
- **Carbonatos**: Calcita (CaCO₃), Dolomita [CaMg(CO₃)₂] — fases de entrampamiento por precipitación de
  carbonatos secundarios
- **Silicatos**: Cuarzo (SiO₂), Feldespatos K (KAlSi₃O₈), Plagioclasas — fuentes de cationes divalentes
  Ca²⁺ y Mg²⁺ para mineralización de CO₂
- **Arcillas**: Caolinita [Al₂Si₂O₅(OH)₄], Illita [K₀.₆Mg₀.₂₅Al₂.₃Si₃.₅O₁₀(OH)₂] — fases que
  controlan la alcalinidad intersticial y la precipitación de carbonatos secundarios
- **Óxidos/sulfuros**: Pirita (FeS₂), Goethita [FeO(OH)] — fases redox activas que influyen el pe y la
  especiación del Fe

In [4]:
# Importación de librerías y configuración del entorno
 
import os
import sys
import warnings
from pathlib import Path
 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
warnings.filterwarnings("ignore")

In [5]:
BASE_DIR   = Path(".").resolve()
DATA_DIR   = BASE_DIR / "data"
DB_DIR     = BASE_DIR / "databases"
OUT_DIR    = BASE_DIR / "outputs"
FIG_DIR    = BASE_DIR / "outputs" / "figures"
 
for d in [DATA_DIR, DB_DIR, OUT_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [6]:
plt.rcParams.update({
    "font.family"        : "DejaVu Sans",
    "font.size"          : 11,
    "axes.spines.top"    : False,
    "axes.spines.right"  : False,
    "axes.grid"          : True,
    "grid.alpha"         : 0.25,
    "grid.linestyle"     : "--",
    "figure.dpi"         : 120,
    "savefig.dpi"        : 200,
})
SAVEFIG_KW = {"bbox_inches": "tight"}
 
PALETTE = {
    "La Luna"   : "#1f77b4",
    "Mugrosa"   : "#ff7f0e",
    "Rosablanca": "#2ca02c",
    "Tablazo y Rosablanca": "#9467bd",
    "Umir"      : "#8c564b",
}
 
print("✓ Entorno configurado correctamente.")
print(f"  Python  {sys.version.split()[0]}")
print(f"  NumPy   {np.__version__}")
print(f"  Pandas  {pd.__version__}")
print(f"  Matplotlib {plt.matplotlib.__version__}")
print(f"\n  Directorio base  : {BASE_DIR}")
print(f"  Figuras → {FIG_DIR}")
 

✓ Entorno configurado correctamente.
  Python  3.13.5
  NumPy   2.1.3
  Pandas  2.2.3
  Matplotlib 3.10.0

  Directorio base  : C:\Users\danna\MODELAMIENTO PROYECTO FINAL
  Figuras → C:\Users\danna\MODELAMIENTO PROYECTO FINAL\outputs\figures


Construcción de la Base de Datos Geoquímica
 
## Nota Metodológica
Este notebook documenta el proceso de recopilación, filtrado y control de calidad de los datos geoquímicos
de aguas de formación y petrografía de roca del VMM. La base de datos fue construida **desde cero** dado
que no existe un repositorio unificado de datos geoquímicos de subsuperficie para esta cuenca en el dominio
público colombiano.
 
---
 
## 1. Metodología de Recopilación de Datos
 
### 1.1 Síntesis Bibliográfica Sistemática
 
La recopilación de datos geoquímicos de aguas de formación y mineralogía de roca del VMM implicó una
búsqueda sistemática en bases de datos científicas (Scopus, Web of Science, SGC-Colombia) realizada por mi utilizando propiedades petrofísicas y parámetros geoquímicos. Se identificaron y revisaron más de 60 referencias entre artículos científicos,
tesis de grado e informes técnicos de la Agencia Nacional de Hidrocarburos (ANH) y el Servicio Geológico
Colombiano (SGC). De estas, fueron seleccionadas aquellas con datos cuantitativos de composición iónica,
pH, conductividad eléctrica, sólidos disueltos totales (TDS) y descripción petrográfica con al menos una
referencia espacial (pozo o campo) dentro del VMM.
 
La información disponible en la literatura resultó fragmentada y con importantes sesgos de publicación:
predominan datos de las formaciones con interés hidrocarburífero activo (La Luna, Mugrosa), mientras que
unidades como Umir, Esmeraldas y la secuencia Tablazo-Rosablanca cuentan con registros geoquímicos
escasos o sin publicar. Esta asimetría condicionó el tamaño de muestra efectivo por formación y explica,
en parte, la variabilidad estadística observada en los datos consolidados.
 
### 1.2 Expedientes del ANLA
 
Una fuente de información de particular relevancia y que demandó un esfuerzo significativo de gestión
y procesamiento fueron los expedientes ambientales radicados ante la Autoridad Nacional de Licencias
Ambientales (ANLA). Los estudios de impacto ambiental (EIA) y planes de manejo ambiental (PMA) de
proyectos de exploración y producción en el VMM contienen, en sus capítulos de caracterización del medio
abiótico, análisis fisicoquímicos de aguas subterráneas profundas y, en algunos casos, de aguas de
formación obtenidas durante pruebas de producción. 
 

El modelado de interacción agua-roca en formaciones de alta salinidad es fundamental para evaluar la compatibilidad de fluidos de inyección, el riesgo de incrustaciones (scale), y los cambios mineralógicos que alteran la permeabilidad del yacimiento. Este informe presenta la especiación geoquímica inicial de cuatro formaciones del bloque de estudio (La Luna, Mugrosa, Rosablanca y Tablazo) y el estado termodinámico de equilibrio alcanzado tras la reacción agua-roca simulada con PHREEQC, usando el comando `EQUILIBRIUM_PHASES` con masa de agua ajustada a la porosidad del yacimiento.

## Escenario Base – Especiación del Agua Cruda

### Composición iónica inicial (mg/L)

| Parámetro       | La Luna | Mugrosa | Rosablanca | Tablazo |
|:--------------- |:-------:|:-------:|:----------:|:-------:|
| Temp. (°C)      | 25.0    | 26.1    | 25.0       | 25.0    |
| pH              | 7.90    | 6.60    | 6.10       | 6.10    |
| TDS est. (mg/L) | ~68,644 | ~26,561 | ~94,600    | ~94,600 |
| Na (mg/L)       | 23,845  | 16,399  | 21,840     | 21,840  |
| K (mg/L)        | 2,410   | 71.4    | 3,805      | 3,805   |
| Ca (mg/L)       | 105.0   | 3,402   | 10,685     | 10,685  |
| Mg (mg/L)       | 62.0    | 202.0   | 1,090      | 1,090   |
| Cl (mg/L)       | 38,510  | 20,123  | 56,265     | 56,265  |
| SO₄ (mg/L)      | 185.0   | 19.5    | 233.8      | 233.8   |
| HCO₃ (mg/L)     | 3,357.5 | 127.6   | 747.2      | 747.2   |
| Fe (mg/L)       | 2.8     | 2.8     | 48.8       | 48.8    |

### Error de balance de carga eléctrica

| Formación  | Error CBE (%) | Interpretación                                                                             |
|:---------- |:-------------:|:------------------------------------------------------------------------------------------ |
| La Luna    | −1.61         | Aceptable (< ±5%)                                                                          |
| Mugrosa    | **+22.87**    | **Alto** – posible submedición de aniones (HCO₃ o acetatos/ácidos orgánicos no reportados) |
| Rosablanca | +2.08         | Aceptable                                                                                  |
| Tablazo    | +2.08         | Aceptable                                                                                  |

> El alto error de Mugrosa (22.87 %) indica que el análisis de laboratorio tiene deficiencia aniónica significativa. Este valor debe ser considerado en la interpretación de los resultados de Mugrosa.

### Índices de saturación (SI) – Escenario Base

Los SI indican la tendencia termodinámica: **SI > 0** → tendencia a precipitar; **SI < 0** → tendencia a disolverse.

| Mineral       | La Luna  | Mugrosa | Rosablanca | Tablazo  |
|:------------- |:--------:|:-------:|:----------:|:--------:|
| Calcita       | **1.22** | 0.05    | **0.65**   | **0.65** |
| Dolomita      | **2.66** | −0.69   | **0.79**   | **0.79** |
| Yeso (Gypsum) | −2.50    | −1.93   | −0.60      | −0.60    |
| Anhidrita     | −2.68    | −2.13   | −0.78      | −0.78    |
| Halita        | −1.90    | −2.33   | −1.75      | −1.75    |
| Siderita      | 0.49     | −0.89   | 0.27       | 0.27     |
| Goethita      | 9.24     | 6.79    | 6.31       | 6.31     |

**Hallazgos clave (agua cruda):**

- **La Luna** presenta la mayor sobresaturación en Calcita (SI = 1.22) y Dolomita (SI = 2.66). Sin la presencia de la roca, el agua tendería a precipitar carbonatos. La alta alcalinidad (3,357 mg/L HCO₃) a pH 7.9 es la responsable.
- **Rosablanca y Tablazo** son idénticas (misma muestra de agua). Moderadamente sobresaturadas en Calcita y Dolomita.
- **Mugrosa** está cercana al equilibrio con Calcita (SI ≈ 0.05), indicando un sistema carbónico casi estabilizado, posiblemente por interacción previa con el yacimiento.
- **Halita, Yeso y Anhidrita** subsaturados en todas las formaciones → no hay riesgo de precipitación de estas fases en el agua cruda.
- **Goethita** fuertemente sobresaturada en todas las formaciones (SI = 6–9), consistente con la presencia de Fe³⁺ oxidado bajo condiciones redox controladas.

---

## Análisis de Interacción Agua-Roca (Equilibrio Termodinámico)

> **Fuente:** `FmSolution_v2.pqi` — archivo de entrada corregido que separa cada formación en una simulación independiente con la directiva `END`, garantizando que PHREEQC ejecute el batch reaction para cada solución + ensamblaje mineral. Se corrigió también la entrada duplicada de Caolinita en Mugrosa. Resultados extraídos de `FmSolution_v2.out`.

### Efecto buffer de pH

| Formación  | pH inicial | pH equilibrio | ΔpH   | Proceso dominante                                                     |
|:---------- |:----------:|:-------------:|:-----:|:--------------------------------------------------------------------- |
| La Luna    | 7.90       | **9.34**      | +1.44 | Buffer silicato-carbonato: disolución Illita + precipitación Dolomita |
| Mugrosa    | 6.60       | **9.79**      | +3.19 | Disolución Illita + precipitación K-feldespato; mayor ΔpH de las 4    |
| Rosablanca | 6.10       | **5.67**      | −0.43 | pH baja: precipitación Calcita consume HCO₃⁻; Pirita en equilibrio    |
| Tablazo    | 6.10       | **5.67**      | −0.43 | Ídem Rosablanca (mineralogía y química idénticas)                     |

**La Luna y Mugrosa** presentan un fuerte buffer alcalino impulsado por la disolución de silicatos alumínicos (Illita). **Rosablanca y Tablazo** responden de forma opuesta: la precipitación de Calcita consume alcalinidad, disminuyendo levemente el pH.

### Transferencia de masa mineral

Δ Moles = Final − Inicial: **(+) precipitación** | **(−) disolución**. Valores ≈ 0 indican mineral prácticamente inerte bajo las condiciones del ensamblaje.

#### La Luna

| Mineral   | Fórmula                      | Δ Moles       | Proceso       | Implicación geoquímica                                          |
|:--------- |:---------------------------- |:-------------:|:-------------:|:--------------------------------------------------------------- |
| Calcita   | CaCO₃                        | **−6.2×10⁻⁴** | Disolución    | Libera Ca²⁺; consecuencia del pH alto y redistribución de CO₃²⁻ |
| Dolomita  | CaMg(CO₃)₂                   | **+8.4×10⁻⁴** | Precipitación | Elimina Ca²⁺ y Mg²⁺ de la solución                              |
| Illita    | K₀.₆Mg₀.₂₅Al₂.₃Si₃.₅O₁₀(OH)₂ | **−2.5×10⁻³** | Disolución    | Fuente de K⁺, Mg²⁺, Al³⁺ y Si                                   |
| Caolinita | Al₂Si₂O₅(OH)₄                | **+2.9×10⁻³** | Precipitación | Consume Al³⁺ y Si liberado por Illita                           |
| Cuarzo    | SiO₂                         | **+3.0×10⁻³** | Precipitación | Consume exceso de Si                                            |

**Reacciones netas:** Illita → Caolinita + Cuarzo + K⁺ (diagénesis tardía) + dolomitización acoplada.

#### Mugrosa

| Mineral      | Fórmula       | Δ Moles       | Proceso       | Implicación geoquímica                      |
|:------------ |:------------- |:-------------:|:-------------:|:------------------------------------------- |
| Calcita      | CaCO₃         | **+4.0×10⁻⁴** | Precipitación | Ligera incrustación carbonática al subir pH |
| Calcedonia   | SiO₂ (amorfo) | **−7.0×10⁻²** | Disolución    | Gran aporte de Si a la solución             |
| Illita       | K₀.₆…O₁₀(OH)₂ | **−1.5×10⁻³** | Disolución    | Libera K⁺, Al³⁺; eleva pH                   |
| K-Feldespato | KAlSi₃O₈      | **+1.1×10⁻³** | Precipitación | Consume K⁺ y Al³⁺ liberados por Illita      |
| Caolinita    | Al₂Si₂O₅(OH)₄ | **+1.2×10⁻³** | Precipitación | Consume Al³⁺ residual                       |
| Cuarzo       | SiO₂          | **+6.9×10⁻²** | Precipitación | Recristalización de Calcedonia → Cuarzo     |

**Reacciones netas:** Transformación polimorfa Calcedonia → Cuarzo (principal por magnitud) + Illita → K-Feldespato + Caolinita. K cae 97% (2.853 → 0.088 mmol/kgw) por captura en K-Feldespato.

#### Rosablanca y Tablazo

| Mineral   | Fórmula       | Δ Moles (Rosablanca) | Δ Moles (Tablazo) | Proceso              |
|:--------- |:------------- |:--------------------:|:-----------------:|:-------------------- |
| Calcita   | CaCO₃         | +6.9×10⁻⁵            | +1.2×10⁻⁴         | Precipitación mínima |
| Caolinita | Al₂Si₂O₅(OH)₄ | ≈ 0                  | ≈ 0               | Inerte               |
| Pirita    | FeS₂          | ≈ 0                  | ≈ 0               | Inerte               |
| Cuarzo    | SiO₂          | ≈ 0                  | ≈ 0               | Inerte               |

**Sistema casi en equilibrio:** el agua de Rosablanca/Tablazo ya está muy próxima al equilibrio con el ensamblaje mineral. La Calcita precipita en trazas (consume HCO₃⁻ → ΔpH = −0.43 unidades). Pirita, Caolinita y Cuarzo son inertes bajo estas condiciones.

### Evolución de concentraciones – todas las formaciones (inicial vs. equilibrio)

#### La Luna

| Ion   | Inicial (mg/L) | Equilibrio (mg/L) | Factor cambio                                               |
|:----- |:--------------:|:-----------------:|:-----------------------------------------------------------:|
| Na    | 23,845         | 23,304            | ≈ 1× (conservativo)                                         |
| K     | 2,410          | 3,051             | +1.27× (enriquecimiento — disolución Illita)                |
| Ca    | 105.0          | **0.72**          | **−146×** (empobrecimiento severo — precipitación Dolomita) |
| Mg    | 62.0           | **0.25**          | **−248×** (empobrecimiento severo — precipitación Dolomita) |
| Cl    | 38,510         | 37,617            | ≈ 1× (conservativo)                                         |
| SO₄   | 185.0          | 180.9             | ≈ 1× (conservativo)                                         |
| HCO₃* | 3,357.5        | 3,757             | +1.12× (leve enriquecimiento)                               |

#### Mugrosa

| Ion   | Inicial (mg/L) | Equilibrio (mg/L) | Factor cambio                                            |
|:----- |:--------------:|:-----------------:|:--------------------------------------------------------:|
| Na    | 16,399         | 16,426            | ≈ 1× (conservativo)                                      |
| K     | 71.4           | **21.2**          | **−3.4×** (empobrecimiento — precipitación K-Feldespato) |
| Ca    | 3,402          | 3,313             | ≈ 1× (leve precipitación Calcita)                        |
| Mg    | 202.0          | 258               | +1.28× (enriquecimiento — disolución Illita)             |
| Cl    | 20,123         | 20,162            | ≈ 1× (conservativo)                                      |
| SO₄   | 19.5           | 19.6              | ≈ 1× (conservativo)                                      |
| HCO₃* | 127.6          | **27.6**          | **−4.6×** (consume alcalinidad al subir pH hasta 9.79)   |

#### Rosablanca

| Ion   | Inicial (mg/L) | Equilibrio (mg/L) | Factor cambio                                                 |
|:----- |:--------------:|:-----------------:|:-------------------------------------------------------------:|
| Na    | 21,840         | 21,230            | ≈ 1× (conservativo)                                           |
| K     | 3,805          | 3,694             | ≈ 1× (conservativo)                                           |
| Ca    | 10,685         | **10,286**        | −1.04× (leve precipitación Calcita)                           |
| Mg    | 1,090          | 1,059             | ≈ 1× (conservativo)                                           |
| Cl    | 56,265         | 54,668            | ≈ 1× (conservativo)                                           |
| SO₄   | 233.8          | 227               | ≈ 1× (conservativo)                                           |
| HCO₃* | 747.2          | 432               | −1.7× (consume alcalinidad — precipitación Calcita + baja pH) |

#### Tablazo

Concentraciones finales idénticas a Rosablanca (misma analítica inicial y mismo ensamblaje mineral). ΔCa = −399 mg/L, ΔHCO₃ = −315 mg/L por precipitación de Calcita (+1.2×10⁻⁴ mol vs +6.9×10⁻⁵ mol de Rosablanca, diferencia proporcional al mayor volumen de agua por porosidad).

*Alcalinidad total expresada como mg/L HCO₃ equivalente (Alk × 61,020 mg/eq × kgw/L).

Na, Cl y SO₄ actúan como **trazadores conservativos** en todas las formaciones, confirmando la integridad de las simulaciones.

In [73]:
# ═══════════════════════════════════════════════════════════════════════════════
# DATOS
# ═══════════════════════════════════════════════════════════════════════════════

# Iones para diagrama de Schoeller
IONS = ["Na", "K", "Ca", "Mg", "Cl", "SO₄", "HCO₃"]

# ── Agua cruda inicial (mg/L) ─────────────────────────────────────────────────
# Fuente: Solution.pqi.out y FmSolution.pqi.out (initial solutions)
# Rosablanca y Tablazo comparten la misma analítica (Tablazo_y_Rosablanca)
INITIAL: dict[str, list[float]] = {
    # fmt: off
    #            Na        K       Ca       Mg       Cl       SO4    HCO3
    "La Luna":    [23845.0, 2410.0,   105.0,   62.0,  38510.0,  185.0, 3357.5],
    "Mugrosa":    [16398.9,   71.4,  3401.8,  202.0,  20122.9,   19.5,  127.6],
    "Rosablanca": [21840.0, 3804.7, 10685.2, 1090.0,  56264.8,  233.8,  747.2],
    "Tablazo":    [21840.0, 3804.7, 10685.2, 1090.0,  56264.8,  233.8,  747.2],
    # fmt: on
}

# ── Agua equilibrada – todas las formaciones (mg/L) ─────────────────────────
# Fuente: FmSolution_v2.out (simulaciones independientes separadas por END)
# Conversión: mg/L = molalidad (mol/kgw) × PM (g/mol) × (kgw/L) × 1000
#   kgw/L = masa_agua_kgw / volumen_solución_L  (tomados del bloque Description)
#
# La Luna  (kgw/L = 0.08223/0.08425 = 0.9760):
#   Na=1.0381×22990×0.976=23304 | K=0.07997×39100×0.976=3051
#   Ca=1.837e-5×40080×0.976=0.72 | Mg=1.035e-5×24310×0.976=0.25
#   Cl=1.0872×35450×0.976=37617 | SO4=1.928e-3×96060×0.976=180.9
#   HCO3=Alk(6.307e-2 eq/kgw)×61020×0.976=3757
# Mugrosa  (kgw/L = 0.1630/0.16504 = 0.9876):
#   Na=0.72395×22990×0.9876=16426 | K=5.489e-4×39100×0.9876=21.2
#   Ca=8.367e-2×40080×0.9876=3313 | Mg=1.074e-2×24310×0.9876=258
#   Cl=0.57606×35450×0.9876=20162 | SO4=2.060e-4×96060×0.9876=19.6
#   HCO3=Alk(4.575e-4)×61020×0.9876=27.6
# Rosablanca / Tablazo  (kgw/L = 0.9711):
#   Na=0.95063×22990×0.9711=21230 | K=9.737e-2×39100×0.9711=3694
#   Ca=2.643e-1×40080×0.9711=10286 | Mg=4.486e-2×24310×0.9711=1059
#   Cl=1.5881×35450×0.9711=54668 | SO4=2.436e-3×96060×0.9711=227
#   HCO3=Alk(7.294e-3)×61020×0.9711=432
EQUILIBRIUM: dict[str, list[float]] = {
    # fmt: off
    #              Na        K       Ca       Mg       Cl       SO4    HCO3
    "La Luna":    [23304.0, 3051.0,  0.72,   0.25,  37617.0,  180.9, 3757.0],
    "Mugrosa":    [16426.0,   21.2, 3313.0, 258.0,  20162.0,   19.6,   27.6],
    "Rosablanca": [21230.0, 3694.0, 10286.0, 1059.0, 54668.0, 227.0,  432.0],
    "Tablazo":    [21230.0, 3694.0, 10286.0, 1059.0, 54668.0, 227.0,  432.0],
    # fmt: on
}

# Alias para compatibilidad con código existente (Fig 2)
LA_LUNA_EQ: list[float] = EQUILIBRIUM["La Luna"]

# ── pH ────────────────────────────────────────────────────────────────────────
# Fuente: Solution.pqi.out (inicial) y FmSolution_v2.out (equilibrio)
PH_INITIAL = {
    "La Luna":    7.90,
    "Mugrosa":    6.60,
    "Rosablanca": 6.10,
    "Tablazo":    6.10,
}
PH_EQ = {
    "La Luna":    9.339,   # pH aumenta por buffer silicato-carbonato
    "Mugrosa":    9.791,   # pH aumenta – disolución K-feldespato + Illita
    "Rosablanca": 5.672,   # pH baja – disolución Pirita produce H+
    "Tablazo":    5.672,   # ídem Rosablanca (mineralogía y química idénticas)
}
PH_EQ_LA_LUNA = PH_EQ["La Luna"]  # retrocompatibilidad

# ── Índice de Saturación – Calcita ────────────────────────────────────────────
# Fuente: FmSolution_v2.out / FmSolution_v2_*.sel (simulaciones independientes)
# Valores exactos del .sel: La Luna=1.2244, Mugrosa=0.0542, Rosablanca=Tablazo=0.6532
# Equilibrio: SI_Calcite = 0.00 confirmado en todos los Phase assemblage blocks
SI_CALCITE = {
    "La Luna":    {"inicial": 1.2244, "equilibrio": 0.00},
    "Mugrosa":    {"inicial": 0.0542, "equilibrio": 0.00},
    "Rosablanca": {"inicial": 0.6532, "equilibrio": 0.00},
    "Tablazo":    {"inicial": 0.6532, "equilibrio": 0.00},
}

# ── Transferencia de masa mineral – La Luna (Δ moles totales) ─────────────────
# Fuente: FmSolution.pqi.out Phase assemblage, Reaction step 1
# Δ = Final − Inicial  →  (+) precipitación  |  (−) disolución
PHASES_DELTA = {
    "Calcita":   -6.213e-4,
    "Dolomita":  +8.354e-4,
    "Illita":    -2.505e-3,
    "Caolinita": +2.881e-3,
    "Cuarzo":    +2.997e-3,
}


In [74]:
# ═══════════════════════════════════════════════════════════════════════════════
# FIGURA 1 – DIAGRAMA DE SCHOELLER
# ═══════════════════════════════════════════════════════════════════════════════
def plot_schoeller() -> None:
    """
    Diagrama semi-logarítmico (Schoeller) comparando la concentración de iones
    mayores en las cuatro formaciones (agua cruda) más el agua de La Luna
    tras equilibrio agua-roca.
    """
    fig, ax = plt.subplots(figsize=(10, 5.8))

    x = np.arange(len(IONS))
    markers = {"La Luna": "o", "Mugrosa": "s", "Rosablanca": "^", "Tablazo": "D"}

    # Aguas crudas iniciales
    for fm, vals in INITIAL.items():
        ax.semilogy(
            x,
            vals,
            linestyle="-",
            marker=markers[fm],
            color=PALETTE[fm],
            linewidth=1.9,
            markersize=7,
            label=f"{fm} (inicial)",
            zorder=3,
        )

    # Líneas de equilibrio agua-roca (discontinuas, mismo color por formación)
    for fm, eq_vals in EQUILIBRIUM.items():
        ax.semilogy(
            x,
            eq_vals,
            linestyle="--",
            marker=markers[fm],
            color=PALETTE[fm],
            linewidth=2.0,
            markersize=6,
            alpha=0.70,
            label=f"{fm} (equilibrio)",
            zorder=4,
        )

    # Flechas en Ca y Mg donde el cambio es más pronunciado (La Luna y Mugrosa)
    for fm in ("La Luna", "Mugrosa"):
        for idx, (ini, eq) in enumerate(zip(INITIAL[fm], EQUILIBRIUM[fm])):
            if IONS[idx] in ("Ca", "Mg") and abs(ini - eq) / max(ini, eq) > 0.3:
                ax.annotate(
                    "",
                    xy=(idx, eq),
                    xytext=(idx, ini),
                    arrowprops=dict(
                        arrowstyle="-|>",
                        color=PALETTE[fm],
                        lw=1.3,
                        alpha=0.55,
                    ),
                )

    ax.set_xticks(x)
    ax.set_xticklabels(IONS, fontsize=12)
    ax.set_ylabel("Concentración (mg/L)", fontsize=12)
    ax.set_title(
        "Diagrama de Schoeller – Aguas de Formación\n"
        "Escenario Base vs. Equilibrio Agua-Roca",
        fontsize=12,
        pad=10,
    )
    ax.set_ylim(0.08, 2e5)
    ax.legend(fontsize=8.5, loc="lower right", framealpha=0.9, ncol=2)

    # Anotación de pH (inicial → equilibrio)
    ph_lines = []
    for fm in ("La Luna", "Mugrosa", "Rosablanca", "Tablazo"):
        ph_lines.append(f"{fm}: pH {PH_INITIAL[fm]:.1f} → {PH_EQ[fm]:.2f}")
    ph_text = "pH  inicial → equilibrio\n" + " | ".join(ph_lines[:2]) + "\n" + " | ".join(ph_lines[2:])
    ax.text(
        0.01, 0.03, ph_text,
        transform=ax.transAxes,
        fontsize=7.5,
        color="dimgray",
        va="bottom",
        style="italic",
    )

    fig.tight_layout()
    path = OUT_DIR / "fig1_schoeller.png"
    fig.savefig(path, **SAVEFIG_KW)
    print(f"  ✓ {path}")
    plot_schoeller()

In [75]:
# ═══════════════════════════════════════════════════════════════════════════════
# FIGURA 2 – GRÁFICO 1:1 (La Luna inicial vs. equilibrio)
# ═══════════════════════════════════════════════════════════════════════════════
def plot_scatter_1to1() -> None:
    """
    Gráfico de dispersión log-log que compara la concentración de iones de
    La Luna antes (eje X) y después (eje Y) de la reacción agua-roca.

    Puntos SOBRE la línea 1:1 → enriquecimiento (ion entró desde la roca)
    Puntos BAJO  la línea 1:1 → empobrecimiento (ion salió hacia la roca)
    """
    ini = np.array(INITIAL["La Luna"])
    eq = np.array(LA_LUNA_EQ)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5),
                              gridspec_kw={"width_ratios": [1.4, 1]})

    # ── Panel izquierdo: scatter 1:1 ─────────────────────────────────────────
    ax = axes[0]

    all_vals = np.concatenate([ini, eq])
    lim_lo = 10 ** (np.floor(np.log10(all_vals.min())) - 0.5)
    lim_hi = 10 ** (np.ceil(np.log10(all_vals.max())) + 0.3)
    lims = [lim_lo, lim_hi]

    # Línea 1:1
    ax.loglog(lims, lims, "k--", linewidth=1.5, alpha=0.55,
              label="Línea 1:1 (sin cambio)")

    # Bandas de factor ×2 / ×0.5
    ax.fill_between(lims, [v * 0.5 for v in lims], [v * 2.0 for v in lims],
                    color="gray", alpha=0.06)

    # Puntos
    OFFSETS = {
        "Na":  (1.15,  1.20),
        "K":   (0.75,  1.25),
        "Ca":  (1.15,  0.55),
        "Mg":  (0.55,  0.55),
        "Cl":  (1.15,  0.80),
        "SO₄": (1.15,  1.25),
        "HCO₃":(0.65,  1.30),
    }

    for ion, c_i, c_f in zip(IONS, ini, eq):
        enrich = c_f >= c_i
        color = "#2ca02c" if enrich else "#d62728"
        ax.scatter(c_i, c_f, s=110, color=color, zorder=5, edgecolors="white",
                   linewidths=0.8)
        ox, oy = OFFSETS.get(ion, (1.2, 1.1))
        ax.annotate(
            ion,
            xy=(c_i, c_f),
            xytext=(c_i * ox, c_f * oy),
            fontsize=9.5,
            fontweight="bold",
            color=color,
            arrowprops=dict(arrowstyle="-", color="lightgray", lw=0.7),
        )

    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_xlabel("Concentración Inicial (mg/L)", fontsize=11)
    ax.set_ylabel("Concentración Equilibrio (mg/L)", fontsize=11)
    ax.set_title("La Luna – Dispersión 1:1\nInicial vs. Equilibrio Agua-Roca", fontsize=11)

    legend_els = [
        mpatches.Patch(facecolor="#2ca02c", label="Enriquecimiento (disolución de roca)"),
        mpatches.Patch(facecolor="#d62728", label="Empobrecimiento (precipitación)"),
        plt.Line2D([0], [0], ls="--", color="k", label="Línea 1:1"),
    ]
    ax.legend(handles=legend_els, fontsize=8.5, loc="upper left", framealpha=0.9)

    # ── Panel derecho: transferencia de masa mineral ──────────────────────────
    ax2 = axes[1]
    phases = list(PHASES_DELTA.keys())
    deltas = list(PHASES_DELTA.values())
    colors_bar = ["#d62728" if d < 0 else "#2ca02c" for d in deltas]
    bars = ax2.barh(phases, [d * 1e3 for d in deltas], color=colors_bar,
                    alpha=0.85, edgecolor="white")
    ax2.axvline(0, color="black", linewidth=1.1, alpha=0.7)
    ax2.set_xlabel("Δ Moles × 10⁻³ (precipitación +, disolución −)", fontsize=9)
    ax2.set_title("Transferencia de Masa\nLa Luna (PHREEQC)", fontsize=11)

    for bar, val in zip(bars, deltas):
        w = bar.get_width()
        offset = 0.03 if w >= 0 else -0.03
        ax2.text(
            w + offset * max(abs(v) * 1e3 for v in deltas),
            bar.get_y() + bar.get_height() / 2,
            f"{val*1e3:+.2f}",
            va="center",
            ha="left" if w >= 0 else "right",
            fontsize=8.5,
        )

    ax2.invert_yaxis()
    ax2.set_xlim(-3.5, 3.5)

    fig.tight_layout(pad=2.0)
    path = OUT_DIR / "fig2_scatter_1to1.png"
    fig.savefig(path, **SAVEFIG_KW)
    print(f"  ✓ {path}")
    plt.close(fig)


In [76]:
# ═══════════════════════════════════════════════════════════════════════════════
# FIGURA 3 – SI CALCITA (barras agrupadas)
# ═══════════════════════════════════════════════════════════════════════════════
def plot_si_calcite() -> None:
    """
    Barras agrupadas que comparan el SI de la Calcita en el Escenario Base
    (agua cruda) vs. el Escenario Equilibrado para las cuatro formaciones.

    Demuestra cómo el equilibrio termodinámico lleva el sistema de
    sobresaturación variable a SI ≈ 0.
    """
    fms = ["La Luna", "Mugrosa", "Rosablanca", "Tablazo"]
    si_ini = [SI_CALCITE[f]["inicial"]    for f in fms]
    si_eq  = [SI_CALCITE[f]["equilibrio"] for f in fms]

    x = np.arange(len(fms))
    w = 0.36

    fig, ax = plt.subplots(figsize=(8.5, 5.2))

    bars_ini = ax.bar(x - w / 2, si_ini, w,
                      label="Agua Cruda (inicial)",
                      color="#4C72B0", alpha=0.88, edgecolor="white", linewidth=0.8)
    bars_eq  = ax.bar(x + w / 2, si_eq, w,
                      label="Agua Equilibrada (PHREEQC target = 0)",
                      color="#55A868", alpha=0.88, edgecolor="white", linewidth=0.8)

    # Etiquetas sobre barras iniciales
    for bar, val in zip(bars_ini, si_ini):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            val + 0.04,
            f"{val:.2f}",
            ha="center", va="bottom", fontsize=10, color="#4C72B0", fontweight="bold",
        )

    # Etiquetas sobre barras de equilibrio
    for bar in bars_eq:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            0.04,
            "0.00",
            ha="center", va="bottom", fontsize=10, color="#2d7a4f", fontweight="bold",
        )

    # Línea de equilibrio
    ax.axhline(0.0, color="crimson", linewidth=1.6, linestyle="--",
               alpha=0.7, label="SI = 0 (equilibrio termodinámico)", zorder=2)

    # Región de subsaturación
    ax.axhspan(-0.5, 0.0, alpha=0.04, color="red")
    ax.axhspan(0.0, 0.2, alpha=0.04, color="green")

    # Anotaciones de zonas
    ax.text(3.7, -0.18, "Subsaturado\n(disolución)", ha="right", fontsize=8,
            color="tomato", style="italic")
    ax.text(3.7,  0.10, "Sobresaturado\n(precipitación)", ha="right", fontsize=8,
            color="seagreen", style="italic")

    ax.set_xticks(x)
    ax.set_xticklabels(fms, fontsize=12)
    ax.set_ylabel("Índice de Saturación – Calcita (log IAP/K)", fontsize=11)
    ax.set_title(
        "Índice de Saturación de Calcita\n"
        "Escenario Base vs. Equilibrio Agua-Roca (PHREEQC)",
        fontsize=12,
        pad=10,
    )
    ax.set_ylim(-0.45, 1.65)
    ax.legend(fontsize=9, loc="upper right", framealpha=0.9)

    ax.annotate(
        "† Equilibrio confirmado en La Luna por output PHREEQC\n"
        "  (Phase assemblage SI = 0.00). Para Mugrosa, Rosablanca\n"
        "  y Tablazo: SI = 0.00 por diseño del modelo.",
        xy=(0.01, 0.02),
        xycoords="axes fraction",
        fontsize=7.8,
        color="dimgray",
        style="italic",
        va="bottom",
    )

    fig.tight_layout()
    path = OUT_DIR / "fig3_si_calcite.png"
    fig.savefig(path, **SAVEFIG_KW)
    print(f"  ✓ {path}")
    plt.close(fig)


# Figura 1. Diagrama de Schoeller

![Figura 1](fig1_schoeller.png)

**Interpretación:** El diagrama semi-logarítmico muestra la "firma química" de cada formación (líneas sólidas) y su estado de equilibrio agua-roca (líneas discontinuas). La Luna y Mugrosa presentan las mayores desviaciones en la dirección alcalina (pH 9.34 y 9.79 respectivamente). El colapso de Ca²⁺ y Mg²⁺ en La Luna hacia valores infra-mg/L es la señal más visible (dolomitización). En Mugrosa, K cae 97% por captura en K-Feldespato. Rosablanca y Tablazo permanecen prácticamente invariantes — confirma que el agua ya está cerca del equilibrio con el ensamblaje Calcita-Caolinita-Cuarzo-Pirita.


# Figura 2. Comparación 1:1

![Figura 2](fig2_scatter_1to1.png)

**Interpretación:** Los puntos sobre la línea 1:1 (verde) indican enriquecimiento por disolución de minerales; los puntos bajo la línea (rojo) indican empobrecimiento por precipitación. Ca y Mg caen drásticamente por debajo de la diagonal — señal inequívoca de precipitación carbonática (dolomitización). K se enriquece por encima de la diagonal, confirmando la disolución de Illita. Na, Cl y SO₄ permanecen en la diagonal, validando su comportamiento conservativo.

# Figura 3. Índice de Saturación de Calcita

![Figura 3](fig3_si_calcite.png)

**Interpretación:** Las barras azules muestran el SI inicial de la Calcita en el agua cruda (La Luna = 1.2244, Rosablanca = Tablazo = 0.6532, Mugrosa = 0.0542). Las barras verdes muestran el SI final tras la reacción con la roca (SI = 0.00 en todas las formaciones, confirmado por PHREEQC en los Phase assemblage blocks). La mayor corrección ocurre en La Luna; Mugrosa ya estaba casi en equilibrio con la Calcita.


## 5. Conclusiones

1. **La Luna** es el agua de formación más reactiva: alta alcalinidad + ensamblaje arcillo-carbonático producen buffer de pH intenso (+1.44 unidades) y dolomitización efectiva que depleta Ca²⁺ (−146×) y Mg²⁺ (−248×).

2. **Mugrosa** muestra el mayor ΔpH (+3.19 unidades, de 6.60 a 9.79) impulsado por la transformación Calcedonia→Cuarzo + disolución de Illita. K cae 97% por precipitación de K-Feldespato. El alto error de balance (22.87%) en el agua cruda debe considerarse al interpretar los índices de saturación iniciales.

3. **Rosablanca y Tablazo** son las salmueras de mayor TDS (~94,600 mg/L) y están casi en equilibrio con el ensamblaje Calcita-Caolinita-Cuarzo-Pirita. Los cambios de equilibrio son mínimos (ΔCa ≈ −400 mg/L, ΔHCO₃ ≈ −315 mg/L) y el pH decrece levemente (−0.43 unidades) por precipitación de Calcita.

4. **Cl⁻, Na⁺ y SO₄²⁻** son trazadores conservativos en todas las formaciones, consistente con la ausencia de fases cloruradas, sulfatadas y sódicas activas en los ensamblajes.

5. La simulación con `FmSolution_v2.pqi` (simulaciones separadas por `END`) confirma que el uso de `EQUILIBRIUM_PHASES` con target SI = 0.00 permite cuantificar la transferencia de masa mineral y predecir la química de yacimiento bajo condiciones in situ para las cuatro formaciones.

# EJERCICIOS EN PHREEQC

## Ejercicio 1 – Solution.pqi (solo especiación, sin minerales)

In [56]:
with open("Solution.pqi") as f:
    lines = f.readlines()

for line in lines[:200]:
    print(line.rstrip())

TITLE Modelacion de Formaciones Geologicas: La Luna, Mugrosa, Tablazo-Rosablanca y Umir

# ---------------------------------------------------------------------
SOLUTION 1 La Luna
    temp            25.0     # Asumido por falta de dato
    pH              7.9
    pe              4.0
    units           mg/L
    density         1.068    # Estimado en base a TDS de ~68,644 mg/L
    Na              23845.0
    Mg              62.0
    Ca              105.0
    Fe              2.8
    Cl              38510.0
    Alkalinity      3357.5 as HCO3 # Suma de HCO3 (3285.0) y CO3 (72.5)
    S(6)            185.0  as SO4
    K               2410.0

SELECTED_OUTPUT 1
    -file           FmSolution/Solution.txt
    -reset          false
    -solution
    -pH
    -alkalinity
    -totals         Na Mg Ca K Cl S(6) C(4) Si Al Fe

END


In [54]:
with open("Solution.pqi.out") as f:
    lines = f.readlines()

for line in lines[:200]:
    print(line.rstrip())

   Input file: .\FmSolution\Solution.pqi
  Output file: .\FmSolution\Solution.pqi.out
Database file: phreeqc.dat

------------------
Reading data base.
------------------

	SOLUTION_MASTER_SPECIES
	SOLUTION_SPECIES
	PHASES
	EXCHANGE_MASTER_SPECIES
	EXCHANGE_SPECIES
	SURFACE_MASTER_SPECIES
	SURFACE_SPECIES
	RATES
	END
------------------------------------
Reading input data for simulation 1.
------------------------------------

	TITLE Modelacion de Formaciones Geologicas: La Luna, Mugrosa, Tablazo-Rosablanca y Umir
	SOLUTION 1 La Luna
	    temp            25.0     # Asumido por falta de dato
	    pH              7.9
	    pe              4.0
	    units           mg/L
	    density         1.068    # Estimado en base a TDS de ~68,644 mg/L
	    Na              23845.0
	    Mg              62.0
	    Ca              105.0
	    Fe              2.8
	    Cl              38510.0
	    Alkalinity      3357.5 as HCO3 # Suma de HCO3 (3285.0) y CO3 (72.5)
	    S(6)            185.0  as SO4
	    K     

# Ejercicio 2 – FmSolution.pqi (La Luna + EQUILIBRIUM_PHASES)

In [58]:
with open("FmSolution.pqi") as f:
    lines = f.readlines()

for line in lines[:200]:
    print(line.rstrip())

TITLE Modelacion de Formaciones: Interaccion Agua-Roca con Porosidad

# =====================================================================
# 1. FORMACION LA LUNA
# =====================================================================
SOLUTION 1 La_Luna
    temp            25.0
    pH              7.9
    pe              4.0
    units           mg/L
    density         1.068
    Na              23845.0
    Mg              62.0
    Ca              105.0
    Fe              2.8
    Cl              38510.0
    Alkalinity      3357.5 as HCO3
    S(6)            185.0  as SO4
    K               2410.0
    -water          0.082269802

EQUILIBRIUM_PHASES 1 Minerales_La_Luna
    Calcite              0.0     0.21639
    Dolomite             0.0     0.00896
    Quartz               0.0     0.08860
    Kaolinite            0.0     0.01991
    Illite               0.0     0.01603


SOLUTION 2 Mugrosa
    temp            25.0
    pH              7.9
    pe              4.0
    units           mg

In [64]:
with open("FmSolution.pqi.out") as f:
    lines = f.readlines()

for line in lines[:200]:
    print(line.rstrip())

   Input file: .\FmSolution\FmSolution.pqi
  Output file: .\FmSolution\FmSolution.pqi.out
Database file: phreeqc.dat

------------------
Reading data base.
------------------

	SOLUTION_MASTER_SPECIES
	SOLUTION_SPECIES
	PHASES
	EXCHANGE_MASTER_SPECIES
	EXCHANGE_SPECIES
	SURFACE_MASTER_SPECIES
	SURFACE_SPECIES
	RATES
	END
------------------------------------
Reading input data for simulation 1.
------------------------------------

	TITLE Modelacion de Formaciones: Interaccion Agua-Roca con Porosidad
	SOLUTION 1 La_Luna
	    temp            25.0
	    pH              7.9
	    pe              4.0
	    units           mg/L
	    density         1.068
	    Na              23845.0
	    Mg              62.0
	    Ca              105.0
	    Fe              2.8
	    Cl              38510.0
	    Alkalinity      3357.5 as HCO3
	    S(6)            185.0  as SO4
	    K               2410.0
	    water          0.082269802
	EQUILIBRIUM_PHASES 1 Minerales_La_Luna
	    Calcite              0.0     0.216

# Ejercicio 3 – FmSolution_v2.pqi 

In [59]:
with open("FmSolution_v2.pqi") as f:
    lines = f.readlines()

for line in lines[:200]:
    print(line.rstrip())

TITLE Modelacion de Formaciones: Interaccion Agua-Roca (4 formaciones independientes)

# =====================================================================
# 1. FORMACION LA LUNA
# =====================================================================
SOLUTION 1 La_Luna
    temp            25.0
    pH              7.9
    pe              4.0
    units           mg/L
    density         1.068
    Na              23845.0
    Mg              62.0
    Ca              105.0
    Fe              2.8
    Cl              38510.0
    Alkalinity      3357.5 as HCO3
    S(6)            185.0  as SO4
    K               2410.0
    -water          0.082269802

EQUILIBRIUM_PHASES 1 Minerales_La_Luna
    Calcite              0.0     0.21639
    Dolomite             0.0     0.00896
    Quartz               0.0     0.08860
    Kaolinite            0.0     0.01991
    Illite               0.0     0.01603

SELECTED_OUTPUT 1
    -file           FmSolution_v2_LLuna.sel
    -reset          false
    -solut

In [53]:
with open("FmSolution_v2.out") as f:
    lines = f.readlines()

for line in lines[:200]:
    print(line.rstrip())

   Input file: FmSolution\FmSolution_v2.pqi
  Output file: FmSolution\FmSolution_v2.out
Database file: phreeqc.dat

------------------
Reading data base.
------------------

	SOLUTION_MASTER_SPECIES
	SOLUTION_SPECIES
	PHASES
	EXCHANGE_MASTER_SPECIES
	EXCHANGE_SPECIES
	SURFACE_MASTER_SPECIES
	SURFACE_SPECIES
	RATES
	END
------------------------------------
Reading input data for simulation 1.
------------------------------------

	TITLE Modelacion de Formaciones: Interaccion Agua-Roca (4 formaciones independientes)
	SOLUTION 1 La_Luna
	    temp            25.0
	    pH              7.9
	    pe              4.0
	    units           mg/L
	    density         1.068
	    Na              23845.0
	    Mg              62.0
	    Ca              105.0
	    Fe              2.8
	    Cl              38510.0
	    Alkalinity      3357.5 as HCO3
	    S(6)            185.0  as SO4
	    K               2410.0
	    water          0.082269802
	EQUILIBRIUM_PHASES 1 Minerales_La_Luna
	    Calcite            

# Interpretaciones

Aunque los tres modelos parten de la misma composición química del agua de formación y, por tanto, generan resultados geoquímicos muy similares, difieren en el nivel de complejidad con el que representan el sistema. El modelo Solution.pqi corresponde a una caracterización básica del agua, permitiendo evaluar la especiación química y los índices de saturación sin considerar reacciones explícitas con minerales. El modelo FmSolution_v2 incorpora fases minerales en equilibrio (calcita, dolomita, cuarzo, caolinita e illita), lo que permite simular procesos de interacción agua-roca y analizar posibles fenómenos de disolución o precipitación mineral. Finalmente, el modelo FmSolution.pqi añade además una restricción de masa de agua asociada al volumen poroso del sistema mediante el parámetro water, haciendo que la simulación sea más representativa de las condiciones reales del reservorio.

Los tres modelos geoquímicos muestran un comportamiento consistente de las aguas de formación analizadas, caracterizadas por una composición altamente salina dominada por Na–Cl, una elevada fuerza iónica (~1,12 mol/kg) y conductividades cercanas a 87.000 µS/cm, valores típicos de salmueras asociadas a sistemas petroleros profundos. La predominancia de Na⁺ y Cl⁻, junto con concentraciones relativamente bajas de Ca²⁺ y Mg²⁺, sugiere una evolución geoquímica avanzada del fluido, posiblemente relacionada con procesos prolongados de interacción agua-roca y concentración de sales durante la historia de enterramiento de las formaciones.ç

La especiación química indica que el carbono inorgánico se encuentra principalmente como bicarbonato (HCO₃⁻), acompañado por complejos carbonatados de sodio, calcio, magnesio y hierro. Esta distribución, combinada con un pH cercano a 7,9, favorece condiciones propicias para la precipitación de minerales carbonatados. En concordancia con ello, los índices de saturación muestran sobresaturación respecto a calcita (SI = 1,22), aragonito (SI = 1,08) y especialmente dolomita (SI = 2,66), evidenciando una tendencia termodinámica hacia la formación o preservación de fases carbonatadas dentro del reservorio.

Por otra parte, las fases evaporíticas presentan índices de saturación negativos, como halita (SI = -1,90), yeso (SI = -2,50) y anhidrita (SI = -2,68), lo que indica que las aguas aún tienen capacidad para disolver estos minerales. Esto sugiere que, aunque se trata de salmueras concentradas, no han alcanzado condiciones de equilibrio con los principales evaporitos presentes en muchas secuencias sedimentarias.
El hierro muestra una fuerte tendencia a la oxidación e inmovilización mineral. Los elevados índices de saturación de goethita (SI = 9,24), hematita (SI = 20,51) y Fe(OH)₃ (SI = 3,34) indican condiciones favorables para la precipitación de óxidos e hidróxidos de hierro. Asimismo, la sobresaturación respecto a siderita (SI = 0,49) sugiere que parte del hierro también podría fijarse en forma de carbonato férrico bajo condiciones adecuadas de disponibilidad de carbono inorgánico.

# Inicio Calibración

In [71]:
with open("OstOutput0_DDS0.txt", "r") as f:
    print(f.read())

--------------------------------------------------------------------------
 OSTRICH version 17.12.19 (Built Dec 19 2017 @ 11:07:05)

 A computer program for model-independent calibration and optimization.

 Author             L. Shawn Matott
 Copyright (C) 2007 L. Shawn Matott

 This program is free software; you can redistribute 
 it and/or modify it under the terms of the GNU  
 General Public License as published by the Free 
 Software Foundation; either version 2 of the 
 License, or(at your option) any later version. 

 This program is distributed in the hope that it will 
 be useful, but WITHOUT ANY WARRANTY; without even 
 the implied warranty of MERCHANTABILITY or FITNESS 
 FOR A PARTICULAR PURPOSE. See the GNU General Public 
 License for more details. 

 You should have received a copy of the GNU General 
 Public License along with this program; if not, 
 write to the Free Software Foundation, Inc., 59 
 Temple Place, Suite 330, Boston, MA 02111-1307 USA 
--------------------

Este es el resultado de una calibración automática realizada con Ostrich (DDS - Dynamically Dimensioned Search) para ajustar los parámetros del modelo hidrogeoquímico.

Lo más importante es el bloque "Optimal Parameter Set", donde aparecen los valores que el algoritmo encontró como la mejor combinación para representar la formación.

Porosidad: 0.318 (31.8%)
Calcita: 0.112 (11.2%)
Dolomita: 0.106 (10.6%)
Cuarzo: 0.634 (63.4%)
Caolinita: 0.203 (20.3%)
Illita: 0.005 (0.5%)

Además, muestra todas las 425 iteraciones que realizó el algoritmo, probando diferentes combinaciones de porosidad y porcentajes minerales para minimizar la función objetivo. Cada fila corresponde a una prueba distinta y permite ver cómo evolucionó la búsqueda de la solución óptima.

En términos geológicos, la solución óptima sugiere una matriz dominada por cuarzo, con contribuciones secundarias de caolinita, calcita y dolomita, mientras que la illita tiene una participación muy baja. La porosidad estimada (~32%) es relativamente alta para una roca consolidada y debería contrastarse con datos petrofísicos o de laboratorio.

In [72]:
import pandas as pd
import subprocess
import math

df = pd.read_csv("ModelParams.txt", sep = '/t', header = 3, names = ['Parameter'], engine = 'python')

Parameters = []
for row in df.iterrows():
    Parameters.append(row[1]["Parameter"][0:7])

x = Parameters
DictMinPerc = {'porosity' : x[0], #average formation porosity 
               'Calcite'  : x[1], #Calcite average percentage
               'Dolomite' : x[2], #Dolomite average percentage
               'Quartz'   : x[3], #Quartz average percetage
               'Kaolinite': x[4], #Kaolinite average percetage
               'Illite'   : x[5]  #Illite average percetage
}
DictMinPerc = {k: float(v) for k, v in DictMinPerc.items()}

mineral_props = {
    'Calcite':   {'rho': 2.71, 'mw': 100.087},
    'Dolomite':  {'rho': 2.84, 'mw': 184.40},
    'Quartz':    {'rho': 2.65, 'mw': 60.08},
    'Kaolinite': {'rho': 2.60, 'mw': 258.16},
    'Illite':    {'rho': 2.75, 'mw': 389.34} 
}

V_total_cm3 = 1000.0
porosity = DictMinPerc['porosity']
V_rock_cm3 = V_total_cm3 * ( 1 - porosity )

mols_mineral = {}

for mineral, props in mineral_props.items():
    if mineral in DictMinPerc:
        fraction = DictMinPerc[mineral]

        V_min_cm3 = V_rock_cm3 * fraction
        mass_g = V_min_cm3 * props['rho']
        mols = mass_g / props['mw']

        mols_mineral[mineral] = mols

with open("FmSolution.pqi", 'r') as f:
    lines = f.readlines()

with open("FmSolution.pqi", 'w') as f:
    for line in lines:
        clean_line = line.strip()
        parts = clean_line.split()

        if len(parts) >= 3 and parts[0] in mols_mineral:
            mineral = parts[0]
            saturation_index = parts[1]
            new_mols_value = mols_mineral[mineral]

            new_line = f"   {mineral:<18} {saturation_index:<7} {new_mols_value:.5f}\n"
            f.write(new_line)
        else:
            f.write(line)

Toma los parámetros que Ostrich está optimizando, los transforma en cantidades mineralógicas físicamente consistentes y actualiza el modelo hidrogeoquímico para cada iteración de calibración. Esto permite que la optimización no se haga sobre moles arbitrarios, sino sobre propiedades geológicas más interpretables.